## Transformação dos dados

Nesta etapa vamos carregar os dados processados e preparar a base para análise.

In [1]:
import pandas as pd


### Carregando os dados

Vamos carregar o aqrquivo Parquet criado na etapa anterior


In [2]:
df= pd.read_parquet("../data/processed/prf_2022_2026.parquet")

In [3]:
df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,405151,2022-01-01,sábado,01:35:00,PI,316,415.0,MARCOLANDIA,Ingestão de álcool pelo condutor,Colisão traseira,...,0,1,1,1,3,-7.432800,-40.682619,SPRF-PI,DEL04-PI,UOP03-DEL04-PI
1,405158,2022-01-01,sábado,02:40:00,PR,116,33.0,CAMPINA GRANDE DO SUL,Ingestão de álcool pelo condutor,Tombamento,...,0,0,1,1,2,-25.114403,-48.846755,SPRF-PR,DEL01-PR,UOP02-DEL01-PR
2,405172,2022-01-01,sábado,05:22:00,MS,163,393.0,NOVA ALVORADA DO SUL,Condutor deixou de manter distância do veículo...,Colisão traseira,...,0,1,1,1,3,-21.228445,-54.456296,SPRF-MS,DEL02-MS,UOP01-DEL02-MS
3,405203,2022-01-01,sábado,07:00:00,RJ,101,457.0,ANGRA DOS REIS,Reação tardia ou ineficiente do condutor,Colisão frontal,...,1,1,0,1,2,-23.031498,-44.177153,SPRF-RJ,DEL03-RJ,UOP03-DEL03-RJ
4,405207,2022-01-01,sábado,09:00:00,MG,40,508.3,RIBEIRAO DAS NEVES,Acumulo de água sobre o pavimento,Saída de leito carroçável,...,0,3,0,0,1,-19.760612,-44.134754,SPRF-MG,DEL02-MG,UOP01-DEL02-MG


### Verificando o tamanho da base

In [4]:
df.shape

(326815, 30)

### Verificando as colunas disponíveis

In [5]:
df.columns.tolist()

['id',
 'data_inversa',
 'dia_semana',
 'horario',
 'uf',
 'br',
 'km',
 'municipio',
 'causa_acidente',
 'tipo_acidente',
 'classificacao_acidente',
 'fase_dia',
 'sentido_via',
 'condicao_metereologica',
 'tipo_pista',
 'tracado_via',
 'uso_solo',
 'pessoas',
 'mortos',
 'feridos_leves',
 'feridos_graves',
 'ilesos',
 'ignorados',
 'feridos',
 'veiculos',
 'latitude',
 'longitude',
 'regional',
 'delegacia',
 'uop']

### Verificando acidentes duplicados

O campo `id` identifica cada ocorrência. Vamos verificar se existem registros duplicados.

In [6]:
df["id"].duplicated().sum()

np.int64(0)

### Verificando os tipos dos dados

In [7]:
df.dtypes

id                                 Int64
data_inversa              datetime64[us]
dia_semana                           str
horario                              str
uf                                   str
br                                 int64
km                               float64
municipio                            str
causa_acidente                       str
tipo_acidente                        str
classificacao_acidente               str
fase_dia                             str
sentido_via                          str
condicao_metereologica               str
tipo_pista                           str
tracado_via                          str
uso_solo                             str
pessoas                            int64
mortos                             int64
feridos_leves                      int64
feridos_graves                     int64
ilesos                             int64
ignorados                          int64
feridos                            int64
veiculos        

### Convertendo a coluna de data

A coluna `data_inversa` será convertida para o tipo datetime.

In [8]:
df["data_inversa"] = pd.to_datetime(
    df["data_inversa"],
    errors="coerce"
)

In [9]:
df["data_inversa"].dtype


dtype('<M8[us]')

### Criando colunas de data

Vamos extrair o ano, mês e dia da data do acidente para facilitar as análises.

In [10]:
df["ano"] = df["data_inversa"].dt.year
df["mes"] = df["data_inversa"].dt.month
df["dia"] = df["data_inversa"].dt.day

In [11]:
df[["data_inversa", "ano", "mes", "dia"]].head()

,data_inversa,ano,mes,dia
0,2022-01-01,2022,1,1
1,2022-01-01,2022,1,1
2,2022-01-01,2022,1,1
3,2022-01-01,2022,1,1
4,2022-01-01,2022,1,1


### Verificando os anos da base

Vamos conferir a quantidade de registros disponíveis para cada ano.

In [12]:
df["ano"].value_counts().sort_index()

ano
2022    64606
2023    67766
2024    73202
2025    72529
2026    48712
Name: count, dtype: int64

### Verificando valores nulos

Vamos analisar quais colunas possuem valores ausentes antes de realizar a limpeza.

In [13]:
df.isnull().sum().sort_values(ascending=False)

id                        0
data_inversa              0
dia_semana                0
horario                   0
uf                        0
br                        0
km                        0
municipio                 0
causa_acidente            0
tipo_acidente             0
classificacao_acidente    0
fase_dia                  0
sentido_via               0
condicao_metereologica    0
tipo_pista                0
tracado_via               0
uso_solo                  0
pessoas                   0
mortos                    0
feridos_leves             0
feridos_graves            0
ilesos                    0
ignorados                 0
feridos                   0
veiculos                  0
latitude                  0
longitude                 0
regional                  0
delegacia                 0
uop                       0
ano                       0
mes                       0
dia                       0
dtype: int64

### Colunas com valores nulos

Vamos exibir somente as colunas que possuem dados ausentes.

In [14]:
nulos = df.isnull().sum()

nulos[nulos > 0].sort_values(ascending=False)

Series([], dtype: int64)

### Verificando registros duplicados

Vamos conferir se existem ocorrências duplicadas através da coluna `id`.

In [15]:
df["id"].duplicated().sum()

np.int64(0)

### Verificando valores das colunas categóricas

Vamos analisar alguns campos de texto para identificar possíveis diferenças de escrita ou padronização.

In [16]:
colunas_texto = [
    "uf",
    "dia_semana",
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "condicao_metereologica",
    "tipo_pista",
    "sentido_via"
]

for coluna in colunas_texto:
    print(f"\n--- {coluna} ---")
    print(df[coluna].value_counts().head(20))


--- uf ---
uf
MG    42485
SC    37562
PR    34983
RJ    27470
RS    23016
SP    21691
BA    18304
GO    15057
PE    14145
MT    11410
ES    11145
PB     8134
MS     7947
RO     6990
RN     6894
PI     6374
CE     6295
MA     5546
DF     4681
PA     4589
Name: count, dtype: int64

--- dia_semana ---
dia_semana
domingo          53269
sábado           52912
sexta-feira      50375
segunda-feira    45366
quinta-feira     42729
quarta-feira     41620
terça-feira      40544
Name: count, dtype: int64

--- tipo_acidente ---
tipo_acidente
Colisão traseira                  63181
Saída de leito carroçável         48691
Colisão transversal               41392
Colisão lateral mesmo sentido     34465
Tombamento                        28189
Colisão com objeto                23161
Colisão frontal                   21851
Queda de ocupante de veículo      14989
Atropelamento de Pedestre         14374
Colisão lateral sentido oposto     9442
Incêndio                           7451
Capotamento             

In [17]:
df[["pessoas", "mortos", "feridos_leves", "feridos_graves", "feridos", "veiculos"]].dtypes

pessoas           int64
mortos            int64
feridos_leves     int64
feridos_graves    int64
feridos           int64
veiculos          int64
dtype: object

### Verificando os tipos das métricas

Vamos conferir se as colunas usadas nas análises quantitativas estão armazenadas como números.

In [18]:
colunas_numericas = [
    "pessoas",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados",
    "feridos",
    "veiculos"
]

df[colunas_numericas].dtypes

pessoas           int64
mortos            int64
feridos_leves     int64
feridos_graves    int64
ilesos            int64
ignorados         int64
feridos           int64
veiculos          int64
dtype: object

### Validando valores numéricos

Vamos verificar se existem valores negativos nas principais métricas da base.

In [20]:
for coluna in colunas_numericas:
    quantidade = (df[coluna] < 0).sum()
    print(f"{coluna}: {quantidade}")

pessoas: 0
mortos: 0
feridos_leves: 0
feridos_graves: 0
ilesos: 0
ignorados: 0
feridos: 0
veiculos: 0


### Validando a consistência das pessoas envolvidas

Vamos verificar se a soma de mortos, feridos, ilesos e ignorados corresponde ao total de pessoas registradas no acidente.

In [21]:
df["soma_pessoas"] = (
    df["mortos"]
    + df["feridos"]
    + df["ilesos"]
    + df["ignorados"]
)

(df["soma_pessoas"] != df["pessoas"]).sum()

np.int64(17597)

In [22]:
df[df["soma_pessoas"] != df["pessoas"]][
    [
        "id",
        "pessoas",
        "mortos",
        "feridos",
        "ilesos",
        "ignorados",
        "soma_pessoas"
    ]
].head()

,id,pessoas,mortos,feridos,ilesos,ignorados,soma_pessoas
9,405254,3,1,0,1,3,5
11,405271,6,0,1,3,3,7
12,405273,2,0,1,0,3,4
17,405338,4,0,1,1,3,5
19,405382,16,1,10,4,4,19


### Validando a quantidade de feridos

Vamos verificar se a coluna `feridos` corresponde à soma de feridos leves e feridos graves.

In [23]:
df["soma_feridos"] = (
    df["feridos_leves"]
    + df["feridos_graves"]
)

(df["soma_feridos"] != df["feridos"]).sum()

np.int64(0)

### Validação de consistência das pessoas

Foi testada a hipótese de que a soma de mortos, feridos, ilesos e ignorados seria igual ao total de pessoas.

Foram encontradas 17.597 ocorrências em que essa igualdade não acontece. Por isso, os valores originais da base serão mantidos sem alteração.

In [24]:
df["diferenca_pessoas"] = df["soma_pessoas"] - df["pessoas"]

df["diferenca_pessoas"].value_counts().sort_index()

diferenca_pessoas
0     309218
1      11004
2       4756
3        889
4        400
5        273
6         69
7         50
8         47
9          7
10        29
11        47
12        12
13         7
14         3
16         1
23         1
48         1
80         1
Name: count, dtype: int64

In [25]:
(df["soma_pessoas"] == df["pessoas"]).sum()

np.int64(309218)

In [26]:
(df["soma_pessoas"] != df["pessoas"]).sum()

np.int64(17597)

In [27]:
df.drop(
    columns=[
        "soma_pessoas",
        "soma_feridos",
        "diferenca_pessoas"
    ],
    inplace=True
)

### Padronizando colunas de texto

Vamos remover espaços extras das colunas categóricas para garantir maior consistência dos dados.

In [28]:
colunas_texto = [
    "uf",
    "dia_semana",
    "municipio",
    "causa_acidente",
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "sentido_via",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "uso_solo"
]

for coluna in colunas_texto:
    df[coluna] = df[coluna].str.strip()

### Conferindo a base transformada

Vamos verificar o tamanho e as primeiras linhas após as transformações.

In [29]:
print("Linhas e colunas:", df.shape)

df.head()

Linhas e colunas: (326815, 33)


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,dia
0,405151,2022-01-01,sábado,01:35:00,PI,316,415.0,MARCOLANDIA,Ingestão de álcool pelo condutor,Colisão traseira,...,1,3,-7.432800,-40.682619,SPRF-PI,DEL04-PI,UOP03-DEL04-PI,2022,1,1
1,405158,2022-01-01,sábado,02:40:00,PR,116,33.0,CAMPINA GRANDE DO SUL,Ingestão de álcool pelo condutor,Tombamento,...,1,2,-25.114403,-48.846755,SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2022,1,1
2,405172,2022-01-01,sábado,05:22:00,MS,163,393.0,NOVA ALVORADA DO SUL,Condutor deixou de manter distância do veículo...,Colisão traseira,...,1,3,-21.228445,-54.456296,SPRF-MS,DEL02-MS,UOP01-DEL02-MS,2022,1,1
3,405203,2022-01-01,sábado,07:00:00,RJ,101,457.0,ANGRA DOS REIS,Reação tardia ou ineficiente do condutor,Colisão frontal,...,1,2,-23.031498,-44.177153,SPRF-RJ,DEL03-RJ,UOP03-DEL03-RJ,2022,1,1
4,405207,2022-01-01,sábado,09:00:00,MG,40,508.3,RIBEIRAO DAS NEVES,Acumulo de água sobre o pavimento,Saída de leito carroçável,...,0,1,-19.760612,-44.134754,SPRF-MG,DEL02-MG,UOP01-DEL02-MG,2022,1,1


In [30]:
df.dtypes

id                                 Int64
data_inversa              datetime64[us]
dia_semana                           str
horario                              str
uf                                   str
br                                 int64
km                               float64
municipio                            str
causa_acidente                       str
tipo_acidente                        str
classificacao_acidente               str
fase_dia                             str
sentido_via                          str
condicao_metereologica               str
tipo_pista                           str
tracado_via                          str
uso_solo                             str
pessoas                            int64
mortos                             int64
feridos_leves                      int64
feridos_graves                     int64
ilesos                             int64
ignorados                          int64
feridos                            int64
veiculos        

### Validação final dos dados

Vamos verificar novamente duplicidades e valores nulos antes de salvar a base transformada.

In [31]:
print("IDs duplicados:", df["id"].duplicated().sum())
print("Valores nulos:", df.isnull().sum().sum())

IDs duplicados: 0
Valores nulos: 0


### Verificando o período dos dados

In [32]:
print("Data inicial:", df["data_inversa"].min())
print("Data final:", df["data_inversa"].max())

Data inicial: 2022-01-01 00:00:00
Data final: 2026-08-31 00:00:00


In [33]:
print("Total de acidentes:", len(df))
print("Total de pessoas:", df["pessoas"].sum())
print("Total de mortos:", df["mortos"].sum())
print("Total de feridos:", df["feridos"].sum())
print("Total de veículos:", df["veiculos"].sum())

Total de acidentes: 326815
Total de pessoas: 849205
Total de mortos: 27395
Total de feridos: 375692
Total de veículos: 651312


### Salvando os dados transformados

Após as etapas de limpeza e validação, vamos salvar a base pronta para análise.

In [34]:
df.to_parquet(
    "../data/processed/prf_2022_2026_clean.parquet",
    index=False
)

In [35]:
df_teste = pd.read_parquet(
    "../data/processed/prf_2022_2026_clean.parquet"
)

df_teste.shape

(326815, 33)